In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/psparks/instacart-market-basket-analysis/products.csv
/kaggle/input/datasets/psparks/instacart-market-basket-analysis/order_products__train.csv
/kaggle/input/datasets/psparks/instacart-market-basket-analysis/orders.csv
/kaggle/input/datasets/psparks/instacart-market-basket-analysis/order_products__prior.csv
/kaggle/input/datasets/psparks/instacart-market-basket-analysis/aisles.csv
/kaggle/input/datasets/psparks/instacart-market-basket-analysis/departments.csv


Notebook Bronze có những checklist sau:

* cài package
* đọc secrets
* đọc CSV từ Kaggle
* check cấu trúc
* upload parquet lên Supabase
* ghi manifest
* test load lại từ Supabase

In [2]:
# Cell 1 — Cài supabase + clone repo
!pip install supabase -q

import subprocess, sys

subprocess.run([
    "git", "clone",
    "https://github.com/rhyni/retail-recommender-system.git",
    "/kaggle/working/repo"
], check=True)

sys.path.insert(0, "/kaggle/working/repo")
print("✓ Repo đã clone xong")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 7.4 MB/s eta 0:00:00


Cloning into '/kaggle/working/repo'...


✓ Repo đã clone xong


In [3]:
# Cell 2 — Đọc Kaggle Secrets → set vào config
from kaggle_secrets import UserSecretsClient
from src import config

secrets = UserSecretsClient()
config.SUPABASE_URL = secrets.get_secret("SUPABASE_URL")
config.SUPABASE_KEY = secrets.get_secret("SUPABASE_KEY")

print("✓ Supabase config loaded")

✓ Supabase config loaded


In [4]:
# Cell 3 — Tạo RUN_ID
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print(f"RUN_ID: {RUN_ID}")

RUN_ID: 20260421T085124Z


In [5]:
# Cell 4a — Chạy Bronze pipeline (có tối ưu RAM)
import pandas as pd
from src import config
from src.io_utils import (
    normalize_columns,
    check_basic_structure,
    optimize_dtypes,
    upload_dataframe,
)
from src.supabase_utils import write_manifest

# Bảng lớn → dùng dtype khi đọc để tiết kiệm RAM ngay từ đầu
# Bảng nhỏ → đọc bình thường
LARGE_TABLES = {
    "order_products__prior",
    "order_products__train",
    "orders",
}

print(f"  BRONZE PIPELINE  |  run_id: {RUN_ID}")

manifest_tables = {}

for table_name, csv_path in config.BRONZE_TABLES.items():
    print(f"\n── {table_name}")

    # bước 2: đọc CSV
    # bảng lớn → truyền dtype vào read_csv để tiết kiệm RAM ngay khi đọc
    # bảng nhỏ → đọc bình thường rồi ép sau cũng được
    if table_name in LARGE_TABLES:
        # lọc ra chỉ những cột có trong DTYPE_MAP
        # các cột không có trong map thì pandas tự suy luận dtype
        df = pd.read_csv(csv_path, dtype=config.DTYPE_MAP)
        print(f"  → đọc với dtype tối ưu (bảng lớn)")
    else:
        df = pd.read_csv(csv_path)
        print(f"  → đọc bình thường (bảng nhỏ)")

    # bước 3: normalize tên cột
    df = normalize_columns(df)

    # bước 4: kiểm tra cấu trúc cơ bản
    check_basic_structure(df, table_name)

    # bước 5: ép kiểu phần còn lại chưa được ép ở read_csv
    # (eval_set → category, các cột nhỏ chưa có trong DTYPE_MAP)
    df = optimize_dtypes(df)

    # bước 6+7+8: chunk → parquet → upload
    upload_dataframe(
        df,
        bucket=config.BRONZE_BUCKET,
        prefix=config.BRONZE_PREFIX,
        run_id=RUN_ID,
        table_name=table_name,
    )

    manifest_tables[table_name] = {"rows": len(df)}

  BRONZE PIPELINE  |  run_id: 20260421T085124Z

── orders
  → đọc với dtype tối ưu (bảng lớn)
  ✓ orders:  3,421,083 rows | 7 cols
  ✓ uploaded [orders]: 7 chunks | 3,421,083 rows

── order_products__prior
  → đọc với dtype tối ưu (bảng lớn)
  ✓ order_products__prior: 32,434,489 rows | 4 cols
  ✓ uploaded [order_products__prior]: 65 chunks | 32,434,489 rows

── order_products__train
  → đọc với dtype tối ưu (bảng lớn)
  ✓ order_products__train:  1,384,617 rows | 4 cols
  ✓ uploaded [order_products__train]: 3 chunks | 1,384,617 rows

── products
  → đọc bình thường (bảng nhỏ)
  ✓ products:     49,688 rows | 4 cols
  ✓ uploaded [products]: 1 file | 49,688 rows

── aisles
  → đọc bình thường (bảng nhỏ)
  ✓ aisles:        134 rows | 2 cols
  ✓ uploaded [aisles]: 1 file | 134 rows

── departments
  → đọc bình thường (bảng nhỏ)
  ✓ departments:         21 rows | 2 cols
  ✓ uploaded [departments]: 1 file | 21 rows


In [6]:
# Cell 4b — Validate trước khi ghi manifest

# 1. Kiểm tra đủ 6 bảng
expected_tables = set(config.BRONZE_TABLES.keys())
uploaded_tables = set(manifest_tables.keys())
assert expected_tables == uploaded_tables, \
    f"Thiếu bảng: {expected_tables - uploaded_tables}"

# 2. Kiểm tra số dòng hợp lý
ROW_CHECKS = {
    "orders":                (3_000_000, 3_500_000),
    "order_products__prior": (30_000_000, 34_000_000),
    "order_products__train": (1_000_000, 2_000_000),
    "products":              (49_000, 50_000),
    "aisles":                (130, 140),
    "departments":           (20, 22),
}

for table, (min_rows, max_rows) in ROW_CHECKS.items():
    actual = manifest_tables[table]["rows"]
    assert min_rows <= actual <= max_rows, \
        f"{table}: {actual:,} dòng — ngoài kỳ vọng [{min_rows:,}, {max_rows:,}]"
    print(f"  ✓ {table}: {actual:,} dòng")

print("\n✅ Tất cả bảng hợp lệ — sẵn sàng ghi manifest")

  ✓ orders: 3,421,083 dòng
  ✓ order_products__prior: 32,434,489 dòng
  ✓ order_products__train: 1,384,617 dòng
  ✓ products: 49,688 dòng
  ✓ aisles: 134 dòng
  ✓ departments: 21 dòng

✅ Tất cả bảng hợp lệ — sẵn sàng ghi manifest


In [7]:
# Cell 5 — Ghi manifest
write_manifest(
    bucket=config.BRONZE_BUCKET,
    prefix=config.BRONZE_PREFIX,
    run_id=RUN_ID,
    layer="bronze",
    created_by="bich",
    tables=manifest_tables,
    source_run_id=None,
)

print(f"  ✅ BRONZE DONE  |  run_id: {RUN_ID}")


  ✓ manifest → bronze-data/instacart/bronze/20260421T085124Z/_manifest.json
  ✅ BRONZE DONE  |  run_id: 20260421T085124Z


In [8]:
# xem các hàm trong file
import src.io_utils as io
dir(io)

['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'check_basic_structure',
 'config',
 'download_parquet',
 'list_files',
 'load_dataframe',
 'math',
 'normalize_columns',
 'optimize_dtypes',
 'pd',
 'upload_dataframe',
 'upload_parquet']

In [9]:
from src.io_utils import load_dataframe

orders = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, RUN_ID, "orders")
products = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, RUN_ID, "products")
aisles = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, RUN_ID, "aisles")
departments = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, RUN_ID, "departments")
prior = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, RUN_ID, "order_products__prior")
train = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, RUN_ID, "order_products__train")

  ✓ loaded [orders]: 7 chunks | 3,421,083 rows
  ✓ loaded [products]: 1 file | 49,688 rows
  ✓ loaded [aisles]: 1 file | 134 rows
  ✓ loaded [departments]: 1 file | 21 rows
  ✓ loaded [order_products__prior]: 65 chunks | 32,434,489 rows
  ✓ loaded [order_products__train]: 3 chunks | 1,384,617 rows


In [10]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [11]:
products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [12]:
aisles.head()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [13]:
departments.head()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [14]:
prior.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [15]:
train.head()

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


In [16]:
from src.supabase_utils import get_client

folder = f"{config.BRONZE_PREFIX}/{RUN_ID}"
items = get_client().storage.from_(config.BRONZE_BUCKET).list(folder)

print("RUN_ID:", RUN_ID)
print("Folder:", folder)
print("Items:")
for x in items:
    print(" -", x["name"])

RUN_ID: 20260421T085124Z
Folder: instacart/bronze/20260421T085124Z
Items:
 - _manifest.json
 - aisles.parquet
 - departments.parquet
 - order_products__prior
 - order_products__train
 - orders
 - products.parquet


In [17]:
import json
from src import config

path = f"{config.BRONZE_PREFIX}/{RUN_ID}/_manifest.json"

data = get_client().storage.from_(config.BRONZE_BUCKET).download(path)
manifest = json.loads(data.decode("utf-8"))

print(json.dumps(manifest, indent=2))

{
  "run_id": "20260421T085124Z",
  "layer": "bronze",
  "created_by": "bich",
  "created_at": "2026-04-21T08:53:01.452466+00:00",
  "tables": {
    "orders": {
      "rows": 3421083
    },
    "order_products__prior": {
      "rows": 32434489
    },
    "order_products__train": {
      "rows": 1384617
    },
    "products": {
      "rows": 49688
    },
    "aisles": {
      "rows": 134
    },
    "departments": {
      "rows": 21
    }
  }
}
